In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
import os
# Define the local folder filename
csv_filename = 'cltpipe_homedepot_data.csv'
# Read the CSV file from the specified path
full_path = os.path.join(r'C:\Users\tobia\OneDrive\Desktop\School\advanced_business_analytics\data', csv_filename)
df = pd.read_csv(full_path)
df.head()
df.count()

month                   1222131
sku name                1222131
sku nbr                 1222131
state territory code    1222131
sub class               1222131
region                  1222131
sales units             1083745
sales $                 1083745
dtype: int64

In [3]:
df_clean = df.drop(columns=['sku name', 'sku nbr', 'state territory code', 'sales $'])
df_clean.head(1)

,month,sub class,region,sales units
0,Fiscal Period 12 of 2025,026-001-005-ABS FITTINGS,0012-PACIFIC NORTHWEST,5.0


In [4]:
df_clean['month_year'] = df_clean['month'].str[-4:]
df_clean.head(1)

,month,sub class,region,sales units,month_year
0,Fiscal Period 12 of 2025,026-001-005-ABS FITTINGS,0012-PACIFIC NORTHWEST,5.0,2025


In [5]:
df_clean['month_number'] = df_clean['month'].str.extract(r'Fiscal Period (\d+)').astype(int)
df_clean.head(1)


,month,sub class,region,sales units,month_year,month_number
0,Fiscal Period 12 of 2025,026-001-005-ABS FITTINGS,0012-PACIFIC NORTHWEST,5.0,2025,12


In [6]:
df_clean = df_clean.drop(columns=['month'])
df_clean['month'] = (df_clean['month_number'].astype(str) + '-' + pd.to_datetime(df_clean['month_year']).dt.strftime('%b-%Y'))
df_clean['month'] = pd.to_datetime('1-' + df_clean['month_number'].astype(str) + '-' + df_clean['month_year'].astype(str), format='%d-%m-%Y')
df_clean = df_clean.drop(columns=['month_year', 'month_number'])
df_clean = df_clean.dropna(subset=['sales units'])
df_clean.head(5)

,sub class,region,sales units,month
0,026-001-005-ABS FITTINGS,0012-PACIFIC NORTHWEST,5.0,2025-12-01
1,026-001-005-ABS FITTINGS,0012-PACIFIC NORTHWEST,-1.0,2026-03-01
2,026-001-005-ABS FITTINGS,0012-PACIFIC NORTHWEST,5.0,2025-02-01
3,026-001-005-ABS FITTINGS,0012-PACIFIC NORTHWEST,3.0,2026-01-01
4,026-001-005-ABS FITTINGS,0012-PACIFIC NORTHWEST,5.0,2025-10-01


In [7]:
df_clean.count()


sub class      1083745
region         1083745
sales units    1083745
month          1083745
dtype: int64

In [8]:
df_transformed = df_clean.copy()
df_transformed= df_transformed.groupby(['region', 'sub class', 'month'])['sales units'].sum().reset_index()
df_transformed.head(5)


,region,sub class,month,sales units
0,0001-MIDSOUTH,026-001-002-PVC PIPE,2023-01-01,10254.0
1,0001-MIDSOUTH,026-001-002-PVC PIPE,2023-02-01,11830.0
2,0001-MIDSOUTH,026-001-002-PVC PIPE,2023-03-01,16240.0
3,0001-MIDSOUTH,026-001-002-PVC PIPE,2023-04-01,14712.0
4,0001-MIDSOUTH,026-001-002-PVC PIPE,2023-05-01,15157.0


In [9]:
df_transformed['sub class'].unique()

<StringArray>
[           '026-001-002-PVC PIPE',        '026-001-003-PVC FITTINGS',
        '026-001-005-ABS FITTINGS',           '026-001-012-CPVC PIPE',
       '026-001-013-CPVC FITTINGS',        '026-001-031-DWV FITTINGS',
            '026-001-032-DWV PIPE',          '026-001-033-SEWER PIPE',
         '026-001-035-DWV PRECUTS',         '026-001-043-PVC PRECUTS',
         '026-001-044-ABS PRECUTS', '026-001-060-S/O PIPE & FITTINGS',
            '026-001-004-ABS PIPE']
Length: 13, dtype: str

In [ ]:
df_transformed['sub class'] = (
    df_transformed['sub class']
    .str.replace('-', '_', regex=False)
    .str.replace(' ', '_', regex=False)
    .str.replace(' &', '', regex=False)
    .str.extract(r'([^_]+_[^_]+)$')[0]
)
df_transformed.head(5)

,region,sub class,month,sales units
0,0001-MIDSOUTH,PVC_PIPE,2023-01-01,10254.0
1,0001-MIDSOUTH,PVC_PIPE,2023-02-01,11830.0
2,0001-MIDSOUTH,PVC_PIPE,2023-03-01,16240.0
3,0001-MIDSOUTH,PVC_PIPE,2023-04-01,14712.0
4,0001-MIDSOUTH,PVC_PIPE,2023-05-01,15157.0


In [11]:
df_transformed.count()

region         7612
sub class      7499
month          7612
sales units    7612
dtype: int64

In [12]:
unique_subclasses = df_transformed['sub class'].unique()
print(unique_subclasses)

<StringArray>
[     'PVC_PIPE',  'PVC_FITTINGS',  'ABS_FITTINGS',     'CPVC_PIPE',
 'CPVC_FITTINGS',  'DWV_FITTINGS',      'DWV_PIPE',    'SEWER_PIPE',
   'DWV_PRECUTS',   'PVC_PRECUTS',   'ABS_PRECUTS',             nan,
      'ABS_PIPE']
Length: 13, dtype: str


In [13]:
df_transformed.count()

region         7612
sub class      7499
month          7612
sales units    7612
dtype: int64

In [14]:
for sub_class in df_transformed['sub class'].dropna().unique():
    globals()[f"df_{str(sub_class).strip().replace(' ', '_').replace('-', '_').replace('/', '_')}"] = df_transformed[df_transformed['sub class'] == sub_class].copy()

In [15]:
[name for name, obj in list(globals().items()) if name.startswith('df_') and name != 'df_transformed' and isinstance(obj, pd.DataFrame)]

['df_clean',
 'df_PVC_PIPE',
 'df_PVC_FITTINGS',
 'df_ABS_FITTINGS',
 'df_CPVC_PIPE',
 'df_CPVC_FITTINGS',
 'df_DWV_FITTINGS',
 'df_DWV_PIPE',
 'df_SEWER_PIPE',
 'df_DWV_PRECUTS',
 'df_PVC_PRECUTS',
 'df_ABS_PRECUTS',
 'df_ABS_PIPE']